In [1]:
# -*- coding: utf-8 -*-
"""
Created on Tue Jul 26 14:42:04 2022

@author: crkpn
"""

import dlib
import scipy.misc
import numpy as np
import os
import cv2
import matplotlib as plt
from gtts import gTTS

import cv2
import matplotlib.pyplot as plt
import pytesseract
import cv2
face_detector = dlib.get_frontal_face_detector()
shape_predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')
face_recognition_model = dlib.face_recognition_model_v1('dlib_face_recognition_resnet_model_v1.dat')
TOLERANCE = 0.6

def get_face_encodings(path_to_image):
    image = cv2.imread(path_to_image)
    detected_faces = face_detector(image, 1)
    shapes_faces = [shape_predictor(image, face) for face in detected_faces]
    return [np.array(face_recognition_model.compute_face_descriptor(image, face_pose, 1)) for face_pose in shapes_faces]

def get_face_encodings1(image):
    detected_faces = face_detector(image, 1)
    shapes_faces = [shape_predictor(image, face) for face in detected_faces]
    return [np.array(face_recognition_model.compute_face_descriptor(image, face_pose, 1)) for face_pose in shapes_faces]

def compare_face_encodings(known_faces, face):
    return (np.linalg.norm(known_faces - face, axis=1) <= TOLERANCE)

def find_match(known_faces, names, face):
    matches = compare_face_encodings(known_faces, face)
    count = 0
    for match in matches:
        if match:
            return names[count]
        count += 1
    return 'Not Found'

image_filenames = filter(lambda x: x.endswith('.jpg'), os.listdir('images/'))
image_filenames = sorted(image_filenames)
paths_to_images = ['images/' + x for x in image_filenames]
face_encodings = []
for path_to_image in paths_to_images:
    face_encodings_in_image = get_face_encodings(path_to_image)
    face_encodings.append(get_face_encodings(path_to_image)[0])
    

file='C:/Users/crkpn/OneDrive/Desktop/darknet-master/build/darknet/x64'
vid = cv2.VideoCapture(0)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
while(True):      
    ret, frame = vid.read()  
    cv2.imshow('frame',frame)  
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    if cv2.waitKey(1) & 0xFF == ord('o'):      
        cv2.imwrite('test.jpg',frame)
        os.system("start cmd /k darknet_no_gpu.exe detect cfg/yolov3.cfg yolov3.weights test.jpg")   
        cv2.waitKey(6000)
        img=cv2.imread('predictions.jpg')
    if cv2.waitKey(1) & 0xFF == ord('f'):      
        face_cascade=cv2.CascadeClassifier('/Users/crkpn/Documents/Programs/opencv-master/data/haarcascades/haarcascade_frontalface_default.xml')
        gray=cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        names = [x[:-4] for x in image_filenames]

        faces=face_cascade.detectMultiScale(gray, 1.3, 5)
        try:
            for (x,y,w,h) in faces:
                face_encodings_in_image = get_face_encodings1(frame)
                # Find match for the face encoding found in this test image
                match = find_match(face_encodings, names, face_encodings_in_image[0])
                print(match)
                if match=='Not Found':
                    ch=input('Do you want to add this face?y/n')
                    if ch=='y' or ch=='Y':
                        name=input('Enter name: ')
                        path='images\\'+name+'.jpg'
                        print(path)
                        cv2.imwrite(path,frame)
                                            
                  
                    else:
                        pass
                myobj = gTTS(match, lang='en', slow=False)
                myobj.save("welcome.mp3")
                os.system("welcome.mp3")
                cv2.waitKey(1000)
                os.remove("welcome.mp3")
        except:
            print('Align Face properly')
        else:
            pass       
    if cv2.waitKey(1) & 0xFF == ord('t'):      
        print(pytesseract.image_to_string(frame))
    if cv2.waitKey(1) & 0xFF == ord('r'): 
        image_filenames = filter(lambda x: x.endswith('.jpg'), os.listdir('images/'))
        image_filenames = sorted(image_filenames)
        paths_to_images = ['images/' + x for x in image_filenames]
        face_encodings = []
        for path_to_image in paths_to_images:
            face_encodings_in_image = get_face_encodings(path_to_image)

            face_encodings.append(get_face_encodings(path_to_image)[0])

vid.release()
cv2.destroyAllWindows()

c:\Users\crkpn\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


RuntimeError: Unable to open shape_predictor_68_face_landmarks.dat